### load sequences

In [12]:
import torch
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore')
import numpy as np
import os
import random
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"       # to force BERT determinsm

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # per multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    
dataset = "MIMIC"
import pickle
with open(f'data/{dataset}_884_event_events.txt','rb') as f:
      event_sequences = pickle.load(f)
sentences = {id: e[0][0] for id,e in event_sequences.items()}
#with open(f'data/{dataset}_visits.txt','rb') as f:
#      visit_sequences = pickle.load(f)
y_df = pd.read_csv(f'data/{dataset}_targets.csv', index_col=0)

In [13]:
from scripts.clinical_text_embedding_pipeline import run_pipeline, run_pipeline_lg
from sklearn.metrics import *
from sklearn.model_selection import StratifiedKFold
from tqdm.notebook import tqdm
import numpy as np

n_splits = 5
cm_total = np.zeros((2, 2), dtype=int)
scores = {
    "mcc": [],
    "precision": [],
    "recall": [],
    "accuracy": [],
    "brier": [],
    "auc": []
}
print(f"\n🚀 Starting {n_splits}-Fold CV...\n")

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
selected_patient_ids = y_df.index.values
y = y_df.values.astype(np.float32).ravel()
cvfolding = tqdm(skf.split(selected_patient_ids, y), total=n_splits, desc="Folds")

for fold, (train_idx, valid_idx) in enumerate(cvfolding):

    print(f"\n========== FOLD {fold} ==========\n")

    train_ids = np.array([selected_patient_ids[i] for i in train_idx])
    valid_ids = np.array([selected_patient_ids[i] for i in valid_idx])

    train_seq = {pid: sentences[pid] for pid in train_ids}
    valid_seq = {pid: sentences[pid] for pid in valid_ids}
    
    y_train = y_df.loc[train_ids].values.astype(np.float32).ravel()
    y_valid = y_df.loc[valid_ids].values.astype(np.float32).ravel()

    # ----------------------------
    # TRAIN
    # ----------------------------
    clf, X_valid, y_valid = run_pipeline_lg(
        train_seq=train_seq,
        valid_seq=valid_seq,
        y_train=y_train,
        y_valid=y_valid,
        mode="lstm"
    )

    # ----------------------------
    # PREDICT
    # ----------------------------
    y_prob = clf.predict(X_valid)
    y_pred = (y_prob > 0.5).astype(int)

    #y_prob = clf.predict_proba(X_valid)[:, 1]
    #y_pred = (y_prob >= 0.5).astype(int)

    # ----------------------------
    # METRICS
    # ----------------------------
    scores["mcc"].append(matthews_corrcoef(y_valid, y_pred))
    scores["precision"].append(precision_score(y_valid, y_pred, zero_division=0))
    scores["recall"].append(recall_score(y_valid, y_pred, zero_division=0))
    scores["accuracy"].append(accuracy_score(y_valid, y_pred))
    scores["brier"].append(brier_score_loss(y_valid, y_prob))
    # ----------------------------
    # CONFUSION MATRIX (FOLD)
    # ----------------------------
    cm = confusion_matrix(y_valid, y_pred)

    print(f"\nConfusion Matrix - Fold {fold}")
    print(cm)

    # accumula
    cm_total += cm
    try:
        scores["auc"].append(roc_auc_score(y_valid, y_prob))
    except:
        scores["auc"].append(np.nan)

    print(f"Fold {fold} MCC: {scores['mcc'][-1]:.4f}")
    print(f"Fold {fold} F1 components → P:{scores['precision'][-1]:.4f} R:{scores['recall'][-1]:.4f}")
    print(f"Fold {fold} ACC: {scores['accuracy'][-1]:.4f}")
    print(f"Fold {fold} Brier: {scores['brier'][-1]:.4f}")
    print(f"Fold {fold} AUC: {scores['auc'][-1]:.4f}")
    
print("\n==============================")
print("FINAL CV RESULTS")
print("==============================")

for k in scores.keys():
    vals = np.array(scores[k], dtype=float)
    print(f"{k.upper():10s} | mean = {np.nanmean(vals):.4f} | std = {np.nanstd(vals):.4f}")
print("\n==============================")
print("CUMULATIVE CONFUSION MATRIX")
print("==============================")
print(cm_total)


🚀 Starting 5-Fold CV...



Folds:   0%|          | 0/5 [00:00<?, ?it/s]


========== FOLD 0 ==========

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's MCC: 0.126622

Confusion Matrix - Fold 0
[[150  11]
 [ 13   3]]
Fold 0 MCC: 0.1266
Fold 0 F1 components → P:0.2143 R:0.1875
Fold 0 ACC: 0.8644
Fold 0 Brier: 0.1174
Fold 0 AUC: 0.5342

========== FOLD 1 ==========

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's MCC: 0.339116

Confusion Matrix - Fold 1
[[161   0]
 [ 14   2]]
Fold 1 MCC: 0.3391
Fold 1 F1 components → P:1.0000 R:0.1250
Fold 1 ACC: 0.9209
Fold 1 Brier: 0.1061
Fold 1 AUC: 0.5846

========== FOLD 2 ==========

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[8]	valid_0's MCC: 0.164713

Confusion Matrix - Fold 2
[[148  12]
 [ 13   4]]
Fold 2 MCC: 0.1647
Fold 2 F1 components → P:0.2500 R:0.2353
Fold 2 ACC: 0.8588
Fold 2 Brier: 0.1185
Fold 2 AUC: 0.4890

========== FOLD 3 ========

In [14]:
y_df.value_counts()

is_alive?
0            802
1             82
Name: count, dtype: int64

In [16]:
event_sequences

{10015988: [('Pneumonia due to other Gram-negative bacteria', '2136-06-01'),
  ('Unspecified severe protein-calorie malnutrition', '2136-06-01'),
  ('Encephalopathy, unspecified', '2136-06-01'),
  ('Chronic systolic (congestive) heart failure', '2136-06-01'),
  ('Hypertensive heart and chronic kidney disease with heart failure and stage 1 through stage 4 chronic kidney disease, or unspecified chronic kidney disease',
   '2136-06-01'),
  ('Delirium due to known physiological condition', '2136-06-01'),
  ('Malignant neoplasm of upper lobe, left bronchus or lung', '2136-06-01'),
  ('Malignant neoplasm of left kidney, except renal pelvis', '2136-06-01'),
  ('Body mass index [BMI] 19.9 or less, adult', '2136-06-01'),
  ('Cachexia', '2136-06-01'),
  ('Cardiomyopathy, unspecified', '2136-06-01'),
  ('Obstructive sleep apnea (adult) (pediatric)', '2136-06-01'),
  ('Gastro-esophageal reflux disease without esophagitis', '2136-06-01'),
  ('Type 2 diabetes mellitus with hypoglycemia without coma'